In [ ]:
##Notebook by AJ Lonski
## Based on notebook from Ye et al. 2025 (GPN-Star)
## Code and commenting optimized by Gemini

# =============================================================================
# 1. IMPORTS AND GLOBAL SETUP
# =============================================================================
!pip install pyfaidx wandb datasets huggingface_hub
import gzip, os, random, shutil, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from pyfaidx import Fasta
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, f1_score, roc_auc_score, precision_recall_curve, roc_curve, auc
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import wandb
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in str(get_ipython())
WORK_DIR = Path("/content/clinvar_cnn_sweep") if IN_COLAB else Path.cwd() / "clinvar_cnn_sweep"
DATA_DIR = WORK_DIR / "data"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
PREDICTIONS_DIR = WORK_DIR / "clinvar/predictions"
for directory in [DATA_DIR, CHECKPOINT_DIR, PREDICTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

HF_DATASET = "songlab/clinvar_vs_benign"
HG38_GZ_PATH = DATA_DIR / "hg38.fa.gz"
HG38_FA_PATH = DATA_DIR / "hg38.fa"
GENOME_URL = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"

SEED = 42
WINDOW_SIZE = 512
MAX_JITTER = int(WINDOW_SIZE * 0.25)
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUC_TO_ID = {"A": 1, "C": 2, "G": 3, "T": 4}
UNK_ID = 0
RC_TABLE = str.maketrans("ACGTN", "TGCAN")


# =============================================================================
# 2. CENTRALIZED HELPER FUNCTIONS
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def ensure_hg38_fasta(gz_path, fa_path, url):
    if not fa_path.exists():
        if not gz_path.exists(): urllib.request.urlretrieve(url, gz_path)
        with gzip.open(gz_path, "rb") as src, open(fa_path, "wb") as dst: shutil.copyfileobj(src, dst)
    return Fasta(str(fa_path), as_raw=True, sequence_always_upper=True)

def clean_variant_df(df):
    df = df.copy()
    df["chrom"] = df["chrom"].astype(str)
    df["ref"] = df["ref"].astype(str).str.upper()
    df["alt"] = df["alt"].astype(str).str.upper()
    df["target"] = (df["label"].astype(str).str.lower().isin(["1", "true", "pathogenic", "likely pathogenic"])).astype(np.int64)
    return df[df["chrom"] != "MT"].drop_duplicates(subset=["chrom", "pos", "ref", "alt"]).reset_index(drop=True)

def infer_score_multiplier(labels, scores):
    labels = pd.Series(labels)
    scores = pd.Series(scores)
    valid = labels.notna() & scores.notna()
    labels = labels[valid]
    scores = scores[valid]
    if len(labels) == 0 or len(np.unique(labels)) < 2: return 1.0
    corr = np.corrcoef(scores, labels)[0, 1]
    return -1.0 if np.isfinite(corr) and corr < 0 else 1.0

def stratified_bootstrap_se(y_true, scores, n_bootstraps=1000, seed=42):
    rng = np.random.RandomState(seed)
    pos_idx, neg_idx = np.where(y_true == 1)[0], np.where(y_true == 0)[0]
    aurocs, auprcs = [], []
    for _ in range(n_bootstraps):
        resampled_idx = np.concatenate([
            rng.choice(pos_idx, size=len(pos_idx), replace=True),
            rng.choice(neg_idx, size=len(neg_idx), replace=True)
        ])
        try:
            aurocs.append(roc_auc_score(y_true.iloc[resampled_idx], scores.iloc[resampled_idx]))
            auprcs.append(average_precision_score(y_true.iloc[resampled_idx], scores.iloc[resampled_idx]))
        except ValueError: continue
    return np.std(aurocs), np.std(auprcs)

# Model Building Helpers
def get_activation(name):
    if name == 'ReLU': return nn.ReLU()
    if name == 'LeakyReLU': return nn.LeakyReLU()
    return nn.GELU()

def get_block(in_c, out_c, kernel, padding, layer_type, dilation=1):
    if layer_type == 'bottleneck':
        mid_c = max(1, out_c // 2)
        return nn.Sequential(
            nn.Conv1d(in_c, mid_c, kernel_size=1), nn.BatchNorm1d(mid_c),
            nn.Conv1d(mid_c, mid_c, kernel_size=kernel, padding=padding, dilation=dilation), nn.BatchNorm1d(mid_c),
            nn.Conv1d(mid_c, out_c, kernel_size=1)
        )
    return nn.Conv1d(in_c, out_c, kernel_size=kernel, padding=padding, dilation=dilation)

class WeightedHuberClassificationLoss(nn.Module):
    def __init__(self, alpha=0.5, delta=1.0):
        super().__init__()
        self.alpha, self.delta = alpha, delta
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        loss = F.huber_loss(probs, targets.float(), reduction='none', delta=self.delta)
        return (loss * torch.where(targets == 1.0, self.alpha, 1.0 - self.alpha)).mean()


# =============================================================================
# 3. DATA EXTRACTION & UNIFIED SPLITTING (RUN ONCE)
# =============================================================================
set_seed(SEED)
fasta = ensure_hg38_fasta(HG38_GZ_PATH, HG38_FA_PATH, GENOME_URL)

class GenomeSequenceContextExtractor:
    def __init__(self, fasta, window_size):
        self.fasta = fasta
        self.window_size = window_size
        self.rc_map = {"A": "T", "C": "G", "G": "C", "T": "A"}

    def fetch_pair(self, chrom, pos, ref, alt, jitter=0):
        chrom_str = str(chrom)
        fasta_chrom = chrom_str[3:] if chrom_str.startswith("chr") else chrom_str
        fasta_chrom = f"chr{fasta_chrom}" if f"chr{fasta_chrom}" in self.fasta.keys() else fasta_chrom
        start = (int(pos) - 1) - self.window_size // 2 + jitter
        end = start + self.window_size
        seq_fwd = self.fasta[fasta_chrom][start:end] if start >= 0 else ("N" * (-start)) + self.fasta[fasta_chrom][0:end]
        seq_fwd = (seq_fwd + ("N" * max(0, self.window_size - len(seq_fwd))))[:self.window_size].upper()
        seq_rev = seq_fwd.translate(RC_TABLE)[::-1]

        def tokenize(s): return np.array([NUC_TO_ID.get(b, UNK_ID) for b in s], dtype=np.int64)
        fwd_ref, rev_ref = tokenize(seq_fwd), tokenize(seq_rev)
        fwd_alt, rev_alt = fwd_ref.copy(), rev_ref.copy()

        center = self.window_size // 2 - jitter
        if 0 <= center < self.window_size:
            fwd_ref[center], fwd_alt[center] = NUC_TO_ID.get(ref, UNK_ID), NUC_TO_ID.get(alt, UNK_ID)
            rev_ref[center], rev_alt[center] = NUC_TO_ID.get(self.rc_map.get(ref, 'N'), UNK_ID), NUC_TO_ID.get(self.rc_map.get(alt, 'N'), UNK_ID)

        return fwd_ref, fwd_alt, rev_ref, rev_alt, center

class SequenceVariantDataset(Dataset):
    def __init__(self, df, extractor, max_jitter=0):
        self.df = df.reset_index(drop=True)
        self.extractor = extractor
        self.max_jitter = max_jitter

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        jitter = random.randint(-self.max_jitter, self.max_jitter) if self.max_jitter > 0 else 0
        try:
            fwd_ref, fwd_alt, rev_ref, rev_alt, _ = self.extractor.fetch_pair(row.chrom, row.pos, row.ref, row.alt, jitter)
            valid = True
        except Exception:
            fwd_ref = fwd_alt = rev_ref = rev_alt = np.zeros(self.extractor.window_size, dtype=np.int64)
            valid = False

        return {
            "fwd_ref_ids": torch.from_numpy(fwd_ref).long(), "fwd_alt_ids": torch.from_numpy(fwd_alt).long(),
            "rev_ref_ids": torch.from_numpy(rev_ref).long(), "rev_alt_ids": torch.from_numpy(rev_alt).long(),
            "label": torch.tensor(float(row.target if "target" in row else 0), dtype=torch.float32),
            "valid": valid
        }

extractor = GenomeSequenceContextExtractor(fasta, WINDOW_SIZE)

# Load the dataset ONCE
print("Loading dataset from Hugging Face...")
raw_df = load_dataset(HF_DATASET, split="test").to_pandas()

# Also download external benchmark predictions ONCE to merge later
print("Downloading SOTA ClinVar predictions...")
snapshot_download(repo_id="songlab/clinvar_vs_benign", local_dir=f"{WORK_DIR}/clinvar", repo_type="dataset")

# Merge external SOTA predictions onto the RAW dataframe before cleaning to ensure indices match
print("Merging external benchmarks...")
available_pred_files = sorted(PREDICTIONS_DIR.glob("*.parquet"))
external_models = []
for pred_path in available_pred_files:
    model_name = pred_path.stem
    temp_df = pd.read_parquet(pred_path)
    if model_name in temp_df.columns: raw_df[model_name] = temp_df[model_name]
    elif "score" in temp_df.columns: raw_df[model_name] = temp_df["score"]
    elif len(temp_df.columns) >= 1: raw_df[model_name] = temp_df.iloc[:, 0]
    external_models.append(model_name)

# Now clean and split the unified dataframe
cleaned_df = clean_variant_df(raw_df)

val_chroms = ["chr21", "21"]
test_chroms = ["chr22", "22"]

val_df = cleaned_df[cleaned_df["chrom"].isin(val_chroms)].reset_index(drop=True)
test_df = cleaned_df[cleaned_df["chrom"].isin(test_chroms)].reset_index(drop=True)
train_df = cleaned_df[~cleaned_df["chrom"].isin(val_chroms + test_chroms)].reset_index(drop=True)

print(f"Training variants: {len(train_df)}")
print(f"Validation variants: {len(val_df)}")
print(f"Held-out Test variants: {len(test_df)}")


# =============================================================================
# 4. ARCHITECTURES
# =============================================================================
class StandardCNN(nn.Module):
    def __init__(self, hidden_dim, dropout, activation_name, layer_type):
        super().__init__()
        act = get_activation(activation_name)
        self.encoder = nn.Sequential(
            get_block(8, hidden_dim, 15, 7, layer_type), act, nn.BatchNorm1d(hidden_dim),
            get_block(hidden_dim, hidden_dim, 9, 4, layer_type), act, nn.BatchNorm1d(hidden_dim),
            get_block(hidden_dim, hidden_dim, 5, 2, layer_type), act
        )
        self.classifier = nn.Sequential(nn.LayerNorm(hidden_dim * 2), nn.Linear(hidden_dim * 2, hidden_dim), act, nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def _encode(self, ref, alt):
        x = torch.cat([F.one_hot(ref.clamp(0,4), 5)[...,1:], F.one_hot(alt.clamp(0,4), 5)[...,1:]], -1).transpose(1, 2).float()
        return self.encoder(x).amax(dim=-1)

    def forward(self, fwd_ref_ids, fwd_alt_ids, rev_ref_ids, rev_alt_ids):
        return {"logits": self.classifier(torch.cat([self._encode(fwd_ref_ids, fwd_alt_ids), self._encode(rev_ref_ids, rev_alt_ids)], -1)).squeeze(-1)}

class ResidualCNN(nn.Module):
    def __init__(self, hidden_dim, dropout, activation_name, layer_type):
        super().__init__()
        act = get_activation(activation_name)
        self.stem = nn.Sequential(get_block(8, hidden_dim, 7, 3, 'standard'), act, nn.BatchNorm1d(hidden_dim))
        self.res1 = nn.Sequential(get_block(hidden_dim, hidden_dim, 5, 2, layer_type), act, nn.BatchNorm1d(hidden_dim))
        self.res2 = nn.Sequential(get_block(hidden_dim, hidden_dim, 5, 2, layer_type), act, nn.BatchNorm1d(hidden_dim))
        self.classifier = nn.Sequential(nn.LayerNorm(hidden_dim * 2), nn.Linear(hidden_dim * 2, hidden_dim), act, nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def _encode(self, ref, alt):
        x = torch.cat([F.one_hot(ref.clamp(0,4), 5)[...,1:], F.one_hot(alt.clamp(0,4), 5)[...,1:]], -1).transpose(1, 2).float()
        x = self.stem(x)
        x = x + self.res1(x)
        x = x + self.res2(x)
        return x.amax(dim=-1)

    def forward(self, fwd_ref_ids, fwd_alt_ids, rev_ref_ids, rev_alt_ids):
        return {"logits": self.classifier(torch.cat([self._encode(fwd_ref_ids, fwd_alt_ids), self._encode(rev_ref_ids, rev_alt_ids)], -1)).squeeze(-1)}

class DilatedCNN(nn.Module):
    def __init__(self, hidden_dim, dropout, activation_name, layer_type):
        super().__init__()
        act = get_activation(activation_name)
        self.encoder = nn.Sequential(
            get_block(8, hidden_dim, 5, 2, layer_type, dilation=1), act, nn.BatchNorm1d(hidden_dim),
            get_block(hidden_dim, hidden_dim, 5, 4, layer_type, dilation=2), act, nn.BatchNorm1d(hidden_dim),
            get_block(hidden_dim, hidden_dim, 5, 8, layer_type, dilation=4), act, nn.BatchNorm1d(hidden_dim),
            get_block(hidden_dim, hidden_dim, 5, 16, layer_type, dilation=8), act
        )
        self.classifier = nn.Sequential(nn.LayerNorm(hidden_dim * 2), nn.Linear(hidden_dim * 2, hidden_dim), act, nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def _encode(self, ref, alt):
        x = torch.cat([F.one_hot(ref.clamp(0,4), 5)[...,1:], F.one_hot(alt.clamp(0,4), 5)[...,1:]], -1).transpose(1, 2).float()
        return self.encoder(x).amax(dim=-1)

    def forward(self, fwd_ref_ids, fwd_alt_ids, rev_ref_ids, rev_alt_ids):
        return {"logits": self.classifier(torch.cat([self._encode(fwd_ref_ids, fwd_alt_ids), self._encode(rev_ref_ids, rev_alt_ids)], -1)).squeeze(-1)}

class InceptionCNN(nn.Module):
    def __init__(self, hidden_dim, dropout, activation_name, layer_type):
        super().__init__()
        act = get_activation(activation_name)
        c1 = c2 = hidden_dim // 3
        c3 = hidden_dim - c1 - c2
        self.conv1 = nn.Conv1d(8, c1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(8, c2, kernel_size=7, padding=3)
        self.conv3 = nn.Conv1d(8, c3, kernel_size=11, padding=5)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.classifier = nn.Sequential(nn.LayerNorm(hidden_dim * 2), nn.Linear(hidden_dim * 2, hidden_dim), act, nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def _encode(self, ref, alt):
        x = torch.cat([F.one_hot(ref.clamp(0,4), 5)[...,1:], F.one_hot(alt.clamp(0,4), 5)[...,1:]], -1).transpose(1, 2).float()
        out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x)], dim=1)
        return F.gelu(self.bn(out)).amax(dim=-1)

    def forward(self, fwd_ref_ids, fwd_alt_ids, rev_ref_ids, rev_alt_ids):
        return {"logits": self.classifier(torch.cat([self._encode(fwd_ref_ids, fwd_alt_ids), self._encode(rev_ref_ids, rev_alt_ids)], -1)).squeeze(-1)}


ARCHITECTURE_MAP = {
    "Standard_CNN": StandardCNN,
    "Residual_CNN": ResidualCNN,
    "Dilated_CNN": DilatedCNN,
    "Inception_CNN": InceptionCNN
}

# =============================================================================
# 5. INITIAL ARCHITECTURE SWEEP
# =============================================================================
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val_auprc', 'goal': 'maximize'},
    'parameters': {
        'architecture':  {'values': list(ARCHITECTURE_MAP.keys())},
        'learning_rate': {'values': [0.001, 0.005]},
        'weight_decay':  {'values': [1e-4, 1e-3, 1e-2]},
        'dropout':       {'values': [0.1, 0.3, 0.5]},
        'hidden_dim':    {'values': [64, 96, 128, 192]},
        'batch_size':    {'values': [16, 32, 64, 128]},
        'huber_delta':   {'values': [0.1, 0.5, 1.0]},
        'layer_type':    {'values': ['standard', 'bottleneck']},
        'optimizer':     {'values': ['AdamW']},
        'activation':    {'values': ['ReLU', 'GELU', 'LeakyReLU']},
        'epochs':        {'value': 10}
    }
}

def train_sweep():
    wandb.init()
    config = wandb.config
    model_class = ARCHITECTURE_MAP[config.architecture]

    model = model_class(
        hidden_dim=config.hidden_dim, dropout=config.dropout,
        activation_name=config.activation, layer_type=config.layer_type
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    alpha = max(1, len(train_df) - train_df["target"].sum()) / len(train_df)
    criterion = WeightedHuberClassificationLoss(alpha=alpha, delta=config.huber_delta)

    loaders = {
        "train": DataLoader(SequenceVariantDataset(train_df, extractor, MAX_JITTER), batch_size=config.batch_size, shuffle=True, num_workers=NUM_WORKERS),
        "val": DataLoader(SequenceVariantDataset(val_df, extractor, 0), batch_size=config.batch_size, num_workers=NUM_WORKERS)
    }

    best_auprc = 0.0
    for epoch in range(config.epochs):
        metrics = {}
        for phase in ["train", "val"]:
            model.train(phase == "train")
            all_probs, all_labels = [], []
            with torch.set_grad_enabled(phase == "train"):
                for batch in loaders[phase]:
                    if not batch["valid"].any(): continue
                    labels = batch["label"].to(DEVICE)
                    logits = model(batch["fwd_ref_ids"].to(DEVICE), batch["fwd_alt_ids"].to(DEVICE), batch["rev_ref_ids"].to(DEVICE), batch["rev_alt_ids"].to(DEVICE))["logits"]
                    loss = criterion(logits, labels)

                    if phase == "train":
                        optimizer.zero_grad()
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()

                    all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            p, l = np.array(all_probs), np.array(all_labels)
            if len(np.unique(l)) > 1:
                metrics.update({
                    f"{phase}_loss": loss.item(),
                    f"{phase}_auprc": average_precision_score(l, p),
                    f"{phase}_auroc": roc_auc_score(l, p)
                })

        wandb.log({"epoch": epoch, **metrics})

        if metrics.get("val_auprc", 0) > best_auprc:
            best_auprc = metrics["val_auprc"]
            torch.save({'state_dict': model.state_dict(), 'config': dict(config), 'val_auprc': best_auprc}, CHECKPOINT_DIR / f"best_{config.architecture}.pt")

sweep_id = wandb.sweep(sweep_config, project="clinvar-cnn-architecture-comparison")
wandb.agent(sweep_id, function=train_sweep, count=20)


In [ ]:
# =============================================================================
# 6. INCEPTION FINE-TUNING SWEEP
# =============================================================================
fine_tune_config = {
    'method': 'bayes',
    'metric': {'name': 'val_auprc', 'goal': 'maximize'},
    'parameters': {
        'learning_rate': {'values': [.001, .005, .01]},
        'weight_decay':  {'values': [.001, .005, .01]},
        'dropout':       {'values': [0.3, 0.4, 0.5]},
        'hidden_dim':    {'values': [64, 128, 192, 256]},
        'batch_size':    {'values': [64, 128, 256]},
        'huber_delta':   {'values': [0.5, 1.0, 1.5]},
        'layer_type':    {'value': 'bottleneck'},
        'optimizer':     {'value': 'AdamW'},
        'activation':    {'values': ['LeakyReLU']},
        'epochs':        {'value': 10}
    }
}

def run_fine_tune():
    wandb.init()
    config = wandb.config
    model = InceptionCNN(
        hidden_dim=config.hidden_dim, dropout=config.dropout,
        activation_name=config.activation, layer_type=config.layer_type
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    alpha = 1.0 - (train_df['target'].sum() / len(train_df))
    criterion = WeightedHuberClassificationLoss(alpha=alpha, delta=config.huber_delta)

    loaders = {
        "train": DataLoader(SequenceVariantDataset(train_df, extractor, MAX_JITTER), batch_size=config.batch_size, shuffle=True, num_workers=NUM_WORKERS),
        "val": DataLoader(SequenceVariantDataset(val_df, extractor, 0), batch_size=config.batch_size, num_workers=NUM_WORKERS)
    }

    best_val_auprc = 0.0
    for epoch in range(config.epochs):
        metrics = {}
        for phase in ["train", "val"]:
            model.train(phase == "train")
            all_probs, all_labels = [], []
            with torch.set_grad_enabled(phase == "train"):
                for batch in loaders[phase]:
                    if not batch["valid"].any(): continue
                    labels = batch["label"].to(DEVICE)
                    logits = model(batch["fwd_ref_ids"].to(DEVICE), batch["fwd_alt_ids"].to(DEVICE), batch["rev_ref_ids"].to(DEVICE), batch["rev_alt_ids"].to(DEVICE))["logits"]
                    loss = criterion(logits, labels)

                    if phase == "train":
                        optimizer.zero_grad()
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()

                    all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            p, l = np.array(all_probs), np.array(all_labels)
            if len(np.unique(l)) > 1:
                metrics.update({
                    f"{phase}_loss": loss.item(),
                    f"{phase}_auprc": average_precision_score(l, p),
                    f"{phase}_auroc": roc_auc_score(l, p)
                })

        wandb.log({"epoch": epoch, **metrics})
        if metrics.get("val_auprc", 0) > best_val_auprc:
            best_val_auprc = metrics["val_auprc"]
            torch.save({'state_dict': model.state_dict(), 'config': dict(config)}, CHECKPOINT_DIR / f"fine_tuned_Inception_CNN.pt")


sweep_id_ft = wandb.sweep(fine_tune_config, project="clinvar-inception-fine-tune")
wandb.agent(sweep_id_ft, function=run_fine_tune, count=10)


# =============================================================================
# 7. FINAL HELD-OUT EVALUATION & BENCHMARKING
# =============================================================================
# Evaluate Fine-Tuned Model directly onto the unified test_df
CHECKPOINT_PATH = CHECKPOINT_DIR / "fine_tuned_Inception_CNN.pt"

if CHECKPOINT_PATH.exists():
    print("Generating predictions for fine-tuned Inception_CNN...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model = InceptionCNN(
        hidden_dim=ckpt['config']['hidden_dim'], dropout=ckpt['config']['dropout'],
        activation_name=ckpt['config']['activation'], layer_type=ckpt['config']['layer_type']
    ).to(DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()

    eval_loader = DataLoader(SequenceVariantDataset(test_df, extractor), batch_size=256, shuffle=False)
    ft_preds = []
    with torch.inference_mode():
        for batch in tqdm(eval_loader, desc="Predicting"):
            logits = model(batch["fwd_ref_ids"].to(DEVICE), batch["fwd_alt_ids"].to(DEVICE), batch["rev_ref_ids"].to(DEVICE), batch["rev_alt_ids"].to(DEVICE))["logits"]
            probs = torch.sigmoid(logits).cpu().numpy()
            ft_preds.extend(np.where(batch["valid"].numpy(), probs, np.nan))

    test_df["Inception_CNN"] = ft_preds

# Infer model signs using val_df for all external models
MANUAL_SIGN_OVERRIDES = {"GPN-MSA": -1.0}
model_signs = {"Inception_CNN": 1.0}

for m in external_models:
    if m in val_df.columns:
        model_signs[m] = infer_score_multiplier(val_df["target"], val_df[m])
model_signs.update(MANUAL_SIGN_OVERRIDES)

# Prepare Final Benchmark dataframe
available_models = [m for m in (["Inception_CNN"] + external_models) if m in test_df.columns]
exclude_cnns = {'Standard_CNN', 'Residual_CNN', 'Dilated_CNN'}

V_final = test_df.dropna(subset=available_models).reset_index(drop=True)
y_true = V_final["target"].astype(int)

print("\nCalculating bootstrapped metrics...")
final_rows = []
for m in available_models:
    scores = V_final[m] * model_signs.get(m, 1.0)
    auroc = roc_auc_score(y_true, scores)
    auprc = average_precision_score(y_true, scores)
    auroc_se, auprc_se = stratified_bootstrap_se(y_true, scores)
    final_rows.append([m, auroc, auprc, auroc_se, auprc_se])

results_df = pd.DataFrame(final_rows, columns=["Model", "AUROC", "AUPRC", "AUROC_se", "AUPRC_se"]).sort_values("AUPRC", ascending=False)
display(results_df)


# =============================================================================
# 8. PLOTTING (AUPRC and AUROC)
# =============================================================================
# Precision-Recall Curve
plt.figure(figsize=(10, 8))
plot_models = [m for m in available_models if m not in exclude_cnns and m != 'GPN-MSA']

for m in plot_models:
    scores = V_final[m] * model_signs.get(m, 1.0)
    precision, recall, _ = precision_recall_curve(y_true, scores)
    auprc = results_df.loc[results_df["Model"] == m, "AUPRC"].values[0]
    linewidth, alpha = (3, 1.0) if m == 'Inception_CNN' else (1.5, 0.7)
    plt.plot(recall, precision, linewidth=linewidth, alpha=alpha, label=f"{m} (AUPRC: {auprc:.3f})")

baseline = y_true.mean()
plt.axhline(baseline, color="gray", linestyle="--", linewidth=1.5, label=f"Baseline ({baseline:.3f})")
plt.xlabel("Recall (Sensitivity)", fontsize=12)
plt.ylabel("Precision (PPV)", fontsize=12)
plt.title("Precision-Recall Curves: Fine-tuned Inception vs. Benchmarks (Held-Out Test)", fontsize=15)
plt.xlim(0.0, 1.0); plt.ylim(0.0, 1.05)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

# ROC Curve
plt.figure(figsize=(10, 8))
for m in plot_models:
    scores = V_final[m] * model_signs.get(m, 1.0)
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc = results_df.loc[results_df["Model"] == m, "AUROC"].values[0]
    linewidth, alpha = (3, 1.0) if m == 'Inception_CNN' else (1.5, 0.7)
    plt.plot(fpr, tpr, linewidth=linewidth, alpha=alpha, label=f"{m} (AUC: {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.5, label='Random Chance (AUC: 0.500)')
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves: Fine-tuned Inception vs. Benchmarks (Held-Out Test)", fontsize=15)
plt.xlim(0.0, 1.0); plt.ylim(0.0, 1.05)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, model_class in ARCHITECTURE_MAP.items():
    model = model_class(hidden_dim=128, dropout=0.0, activation_name='ReLU', layer_type='standard')
    print(f"{name:>15}: {count_parameters(model):>7,} parameters")

print("\nExact parameter count for the Fine-Tuned Inception_CNN checkpoint:")
ckpt_path = CHECKPOINT_DIR / "fine_tuned_Inception_CNN.pt"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = ckpt['config']
    ft_model = InceptionCNN(
        hidden_dim=cfg['hidden_dim'],
        dropout=cfg['dropout'],
        activation_name=cfg['activation'],
        layer_type=cfg['layer_type']
    )
    print(f"{name:>15}: {count_parameters(ft_model):>7,} parameters (hidden_dim={cfg['hidden_dim']})")
else:
    print("Fine-tuned checkpoint not found.")

In [ ]:
# =============================================================================
# 9. HYPERPARAMETER & METRICS AUDIT TABLE
# =============================================================================
import torch
import pandas as pd
from pathlib import Path

# Define the models, their checkpoint paths, and their names in the results_df
audit_targets = [
    ("Standard_CNN", CHECKPOINT_DIR / "best_Standard_CNN.pt", "Standard_CNN"),
    ("Residual_CNN", CHECKPOINT_DIR / "best_Residual_CNN.pt", "Residual_CNN"),
    ("Dilated_CNN", CHECKPOINT_DIR / "best_Dilated_CNN.pt", "Dilated_CNN"),
    ("Inception_CNN (Orig)", CHECKPOINT_DIR / "best_Inception_CNN.pt", None), # Orig wasn't run on Test
    ("Inception_CNN (Fine-Tuned)", CHECKPOINT_DIR / "fine_tuned_Inception_CNN.pt", "Inception_CNN")
]

audit_data = []

for display_name, ckpt_path, test_df_name in audit_targets:
    if not ckpt_path.exists():
        print(f"Skipping {display_name}: Checkpoint not found at {ckpt_path.name}")
        continue

    # 1. Load the configuration saved by WandB inside the .pt file
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    config = ckpt.get('config', {})

    # 2. Fetch Test Metrics from the final results dataframe
    test_auprc, test_auroc = "N/A", "N/A"
    if test_df_name and 'results_df' in globals():
        row = results_df[results_df['Model'] == test_df_name]
        if not row.empty:
            test_auprc = f"{row['AUPRC'].values[0]:.4f}"
            test_auroc = f"{row['AUROC'].values[0]:.4f}"

    # 3. Append to our audit table
    audit_data.append({
        "Architecture": display_name,
        "Test AUPRC": test_auprc,
        "Test AUROC": test_auroc,
        "Hidden Dim": config.get('hidden_dim', 'N/A'),
        "Layer Type": config.get('layer_type', 'N/A'),
        "Activation": config.get('activation', 'N/A'),
        "Learning Rate": config.get('learning_rate', 'N/A'),
        "Weight Decay": config.get('weight_decay', 'N/A'),
        "Dropout": config.get('dropout', 'N/A'),
        "Batch Size": config.get('batch_size', 'N/A'),
        "Huber Delta": config.get('huber_delta', 'N/A')
    })

# 4. Generate and display the clean table
audit_df = pd.DataFrame(audit_data)
display(audit_df)

In [ ]:
import wandb

api = wandb.Api()
sweep = api.sweep(f"ajlonski-the-university-of-chicago/clinvar-cnn-architecture-comparison/{sweep_id}")

print(f"{'Model':>15} | {'Val AUPRC':>10} | {'Val AUROC':>10}")
print("-" * 43)

for model_name in ARCHITECTURE_MAP.keys():
    runs = [r for r in sweep.runs if r.config.get('architecture') == model_name]
    if not runs:
        print(f"{model_name:>15} | {'N/A':>10} | {'N/A':>10}")
        continue

    best_run = sorted(runs, key=lambda r: r.summary.get('val_auprc', 0), reverse=True)[0]
    val_auprc = best_run.summary.get('val_auprc', 0)
    val_auroc = best_run.summary.get('val_auroc', 0)
    print(f"{model_name:>15} | {val_auprc:10.4f} | {val_auroc:10.4f}")
